# OCR

Builds and updates OCR cache in `data/notebook/ocr.sqlite3` and writes OCR performance to
`data/notebook/perf/ocr.csv`.

In [ ]:
import hashlib
import json
import sqlite3
import time
from pathlib import Path

import pandas as pd
import pytesseract
from PIL import Image
from tqdm import tqdm

from bruki.config import load_config, resolve_paths

Setup paths and runtime toggles.
        


In [ ]:
ROOT = Path.cwd().parent

LABELS_PATH = ROOT / "data/notebook/labels.jsonl"
DB_PATH     = ROOT / "data/notebook/ocr.sqlite3"
PERF_CSV    = ROOT / "data/notebook/perf/ocr.csv"

OCR_PSM    = 6
OCR_OEM    = 3
PERF_N     = 100
RERUN_PERF = False

DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PERF_CSV.parent.mkdir(parents=True, exist_ok=True)

Resolve full corpus image paths and labeled paths.
        


In [ ]:
config = load_config(str(ROOT / "config.yaml"))
path_groups = resolve_paths(config, series=config.plots["screenshots"].series, prefix="screenshot")
all_image_paths = [path for _, _, source_paths in path_groups for path in source_paths]
all_image_paths = list(dict.fromkeys(all_image_paths))

rows = [json.loads(line) for line in LABELS_PATH.read_text(
    encoding="utf-8").splitlines() if line.strip()]
labeled_paths = [ROOT / row["input_path"] for row in rows]

ocr_paths = list(dict.fromkeys(labeled_paths + all_image_paths))

print(f"full corpus paths: {len(all_image_paths)}")
print(f"labeled paths:     {len(labeled_paths)}")
print(f"ocr target paths:  {len(ocr_paths)}")

Populate/update OCR sqlite cache.
        


In [ ]:
def ocr_image(path: Path, psm: int = OCR_PSM, oem: int = OCR_OEM) -> str:
    with Image.open(path) as img:
        if img.size[0] < 10 or img.size[1] < 10:
            return ""
        return pytesseract.image_to_string(
            img.convert("RGB"),
            lang="eng",
            config=f"--psm {psm} --oem {oem}",
        )


conn = sqlite3.connect(DB_PATH)
conn.execute(
    "CREATE TABLE IF NOT EXISTS ocr_doc ("
    "input_path TEXT PRIMARY KEY, "
    "text TEXT NOT NULL)"
)

known_paths = {input_path for (input_path,) in conn.execute("SELECT input_path FROM ocr_doc")}
new_docs = []
for path in tqdm(ocr_paths, desc="ocr"):
    input_path = str(path)
    if input_path in known_paths:
        continue
    new_docs.append((input_path, ocr_image(path).lower()))

if new_docs:
    with conn:
        conn.executemany("INSERT INTO ocr_doc(input_path, text) VALUES (?, ?)", new_docs)

ocr_by_path = dict(conn.execute("SELECT input_path, text FROM ocr_doc"))
conn.close()

print(f"ocr complete: {len(ocr_by_path)} rows (new: {len(new_docs)})")

Compute OCR throughput on stable sampled paths and write `perf/ocr.csv`.

In [ ]:
def stable_sample_paths(paths: list[Path], n: int, salt: str = "perf-v1") -> list[Path]:
    if n < 0:
        return paths
    scored = sorted(
        paths,
        key=lambda p: hashlib.sha256(f"{salt}:{p.as_posix()}".encode()).hexdigest(),
    )
    return scored[: min(n, len(scored))]


def probe_ocr_rate(
    image_paths: list[Path],
    n: int = PERF_N,
    psm: int = OCR_PSM,
    oem: int = OCR_OEM,
):
    probe = stable_sample_paths(image_paths, n)
    skipped = 0
    t0 = time.perf_counter()
    for path in tqdm(probe, desc="ocr-perf"):
        try:
            _ = ocr_image(path, psm=psm, oem=oem)
        except OSError:
            skipped += 1
    elapsed       = time.perf_counter() - t0
    processed     = len(probe) - skipped
    sec_per_image = elapsed / max(processed, 1)
    est_full_min  = (sec_per_image * len(image_paths)) / 60.0
    return {
        "model": "tesseract-ocr-psm6-oem3",
        "family": "ocr",
        "backend": "tesseract",
        "speed_vs_fastest_x": 1.0,
        "sec_per_image": sec_per_image,
        "est_full_min": est_full_min,
        "est_full_hr": est_full_min / 60.0,
        "sample_n": len(probe),
        "processed_n": processed,
        "skipped_n": skipped,
        "elapsed_s": elapsed,
    }


def ocr_perf_results(n: int = PERF_N, rerun: bool = RERUN_PERF) -> pd.DataFrame:
    if PERF_CSV.exists() and not rerun:
        return pd.read_csv(PERF_CSV)

    row = probe_ocr_rate(all_image_paths, n=n)
    df = pd.DataFrame([row])
    df.to_csv(PERF_CSV, index=False)
    return df


perf_out = ocr_perf_results()